# Data Prep & Transformation | Aula 2 · Prática
## Lendo e reconstruindo o modelo transacional da Olist

**Luciano Gillieron Gavinho, PhD.** · MBA em Engenharia de Dados · Universidade Presbiteriana Mackenzie

Na Aula 1 você diagnosticou os arquivos brutos da Olist. Muitos dos problemas encontrados não são acidentes: são consequências de **decisões de modelagem** do sistema transacional, ou da falta delas. Hoje você vai fazer **engenharia reversa** do modelo (descobrir, a partir dos dados, como ele foi desenhado), testar se ele respeita as **formas normais**, impor **restrições** que os arquivos CSV não têm e, por fim, remodelar o pedido como **documento JSON**, no estilo de um banco NoSQL.

**Como trabalhar**
- Os mesmos grupos da Aula 1. Cada pessoa roda o próprio notebook, e o grupo consolida as respostas.
- Comece pela **seção A · Aquecimento**: exemplos resolvidos com a sintaxe nova de hoje.
- O produto da prática é o **Modelo do Grupo**, no fim do notebook.

### Termos usados neste notebook

| Termo | O que significa |
|---|---|
| **Entidade** | Coisa do negócio sobre a qual se guardam dados: cliente, pedido, produto. Vira uma tabela. |
| **Atributo** | Característica de uma entidade: cidade do cliente, preço do item. Vira uma coluna. |
| **Relacionamento** | Ligação entre entidades: um pedido *tem* itens. |
| **Cardinalidade** | Quantas ocorrências de uma entidade se ligam a uma ocorrência da outra: **1:1** (um para um), **1:N** (um para muitos), **N:N** (muitos para muitos). |
| **PK** | *Primary Key*, chave primária: coluna ou conjunto de colunas que identifica cada linha, sem repetição e sem nulo. |
| **FK** | *Foreign Key*, chave estrangeira: coluna que aponta para a PK de outra tabela. |
| **Chave composta** | PK formada por mais de uma coluna, como (`order_id`, `order_item_id`). |
| **Restrição** (*constraint*) | Regra que o banco de dados impõe e verifica sozinho: PK, FK, `NOT NULL` (não pode ser nulo), `UNIQUE` (não pode repetir). |
| **DDL** | *Data Definition Language*: comandos SQL que criam estruturas, como `CREATE TABLE`. |
| **Dependência funcional** (X → Y) | Para cada valor de X existe um único valor de Y. Ex.: `order_id → order_status`. |
| **Normalização** | Organizar as tabelas para que cada fato fique guardado num só lugar, evitando redundância e anomalias. |
| **Formas normais** (1FN, 2FN, 3FN) | Níveis de normalização. Cada uma proíbe um tipo de redundância (explicadas no material). |
| **Anomalia de atualização** | Inconsistência que surge quando o mesmo fato está guardado em vários lugares e só alguns são alterados. |
| **NoSQL** | Família de bancos não relacionais: documento, chave-valor, colunar e grafo. |
| **JSON** | *JavaScript Object Notation*: formato de texto que guarda dados aninhados, com a estrutura junto do dado. |
| **Documento** | Registro autocontido em JSON, que pode **embutir** dados relacionados (os itens dentro do pedido). |
| **STRUCT / LIST** | Tipos do DuckDB: `STRUCT` é um registro com campos nomeados; `LIST` é uma lista de valores. Equivalem a objeto e lista em JSON. |

## 0 · Preparação do ambiente
Execute as duas células abaixo. A primeira baixa os dados do repositório da Olist com `git clone` (cópia do repositório) e instala o DuckDB com `pip` (instalador de pacotes Python). A segunda registra cada arquivo CSV como uma *view* (consulta salva com nome) e cria a função `sql()`, que executa uma consulta e devolve um DataFrame (tabela do pandas).

In [1]:
import os, subprocess
if not os.path.exists("work-at-olist-data"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/olist/work-at-olist-data.git"], check=True)
subprocess.run(["pip", "install", "-q", "duckdb"], check=False)
print(sorted(os.listdir("work-at-olist-data/datasets")))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:
import duckdb, pandas as pd, json
DATA = "work-at-olist-data/datasets"
con = duckdb.connect()

FONTES = {
    "customers":  "olist_customers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
for nome, arq in FONTES.items():
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv_auto('{DATA}/{arq}', header=true)")

def sql(q):
    """Executa uma consulta no DuckDB e devolve um DataFrame."""
    return con.sql(q).df()

def tentar(comando):
    """Executa um comando SQL e, se o banco recusar, mostra o erro em vez de interromper o notebook."""
    try:
        con.execute(comando)
        print("OK: comando executado.")
    except Exception as e:
        print("RECUSADO PELO BANCO:", type(e).__name__)
        print(str(e)[:300])

sql("SELECT COUNT(*) AS pedidos FROM orders")

,pedidos
0,99441


## A · Aquecimento: exemplos resolvidos
Cinco exemplos prontos com a sintaxe nova de hoje. Execute e leia o resultado. Eles usam tabelas pequenas ou a tabela `sellers` (vendedores) e **não** respondem aos exercícios.

### A.1 · Medir uma cardinalidade
Quantos pagamentos tem cada pedido? Primeiro contamos por pedido; depois resumimos com mínimo, máximo e média. O máximo diz se o relacionamento é 1:1 ou 1:N.

In [3]:
sql("""
WITH por_pedido AS (
    SELECT order_id, COUNT(*) AS pagamentos
    FROM payments
    GROUP BY order_id
)
SELECT MIN(pagamentos) AS minimo, MAX(pagamentos) AS maximo, ROUND(AVG(pagamentos), 2) AS media
FROM por_pedido
""")

,minimo,maximo,media
0,1,29,1.04


### A.2 · Testar uma dependência funcional
A dependência X → Y vale quando nenhum valor de X aparece com mais de um valor de Y. O teste é sempre o mesmo: agrupar por X e contar quantos grupos têm mais de um Y distinto. Resultado 0 significa que a dependência vale **nos dados atuais** (o que não prova que valha como regra do negócio).

Aqui testamos `seller_id → seller_state`: cada vendedor tem um único estado?

In [4]:
sql("""
SELECT COUNT(*) AS violacoes
FROM (
    SELECT seller_id
    FROM sellers
    GROUP BY seller_id
    HAVING COUNT(DISTINCT seller_state) > 1
)
""")

,violacoes
0,0


### A.3 · Criar uma tabela com restrições
`CREATE TABLE` define a estrutura e as regras. Abaixo, uma tabela de exemplo com PK e `NOT NULL`. Depois tentamos inserir uma linha que viola a PK: o banco recusa. A função `tentar()` (criada na seção 0) mostra o erro sem interromper o notebook.

In [5]:
con.execute("CREATE OR REPLACE TABLE exemplo_uf (uf VARCHAR PRIMARY KEY, nome VARCHAR NOT NULL)")
tentar("INSERT INTO exemplo_uf VALUES ('SP', 'São Paulo'), ('RJ', 'Rio de Janeiro')")
tentar("INSERT INTO exemplo_uf VALUES ('SP', 'Outro nome para SP')")   # PK repetida
tentar("INSERT INTO exemplo_uf VALUES ('MG', NULL)")                   # nome nulo
sql("SELECT * FROM exemplo_uf")

OK: comando executado.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: Duplicate key "uf: SP" violates primary key constraint.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: NOT NULL constraint failed: exemplo_uf.nome


,uf,nome
0,SP,São Paulo
1,RJ,Rio de Janeiro


### A.4 · Uma FK em ação
A tabela `exemplo_cidade` aponta para `exemplo_uf` pela FK `uf`. Inserir uma cidade com UF existente funciona; com UF inexistente, o banco recusa. É a **integridade referencial** sendo garantida pelo próprio banco.

In [6]:
con.execute("""
CREATE OR REPLACE TABLE exemplo_cidade (
    cidade VARCHAR PRIMARY KEY,
    uf     VARCHAR NOT NULL REFERENCES exemplo_uf(uf)
)""")
tentar("INSERT INTO exemplo_cidade VALUES ('Campinas', 'SP')")
tentar("INSERT INTO exemplo_cidade VALUES ('Manaus', 'AM')")   # AM não existe em exemplo_uf
sql("SELECT * FROM exemplo_cidade")

OK: comando executado.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: Violates foreign key constraint because key "uf: AM" does not exist in the referenced table


,cidade,uf
0,Campinas,SP


### A.5 · De linhas para documento JSON, e de volta
`struct_pack(campo := valor, ...)` monta um registro com campos nomeados; `list(...)` junta várias linhas numa lista; `to_json(...)` converte para texto JSON. Para ler de volta, `json_extract_string(doc, '$.campo')` extrai um valor e `json_array_length(doc, '$.lista')` conta os elementos de uma lista.

Abaixo, um documento por estado com a lista de cidades dos vendedores daquele estado (limitado a 3 cidades, para caber na tela).

In [7]:
docs = sql("""
SELECT seller_state,
       to_json(struct_pack(
           uf      := seller_state,
           cidades := list(DISTINCT seller_city ORDER BY seller_city)[1:3]
       )) AS doc
FROM sellers
WHERE seller_state IN ('AM', 'MA')
GROUP BY seller_state
""")
display(docs)
print(json.dumps(json.loads(docs.loc[0, "doc"]), indent=2, ensure_ascii=False))

con.register("docs_exemplo", docs)   # registra o DataFrame como tabela para consultar em SQL
sql("""
SELECT json_extract_string(doc, '$.uf') AS uf,
       json_array_length(doc, '$.cidades') AS qtd_cidades
FROM docs_exemplo
""")

,seller_state,doc
0,MA,"{""uf"":""MA"",""cidades"":[""sao luis""]}"
1,AM,"{""uf"":""AM"",""cidades"":[""manaus""]}"


{
  "uf": "MA",
  "cidades": [
    "sao luis"
  ]
}


,uf,qtd_cidades
0,MA,1
1,AM,1


In [8]:
def contar(q):
    """Devolve o primeiro valor de uma consulta (útil para contagens)."""
    return con.sql(q).fetchone()[0]

> **Resumo das ferramentas:** cardinalidade com `GROUP BY` + `MIN/MAX`; dependência funcional com `HAVING COUNT(DISTINCT y) > 1`; restrições com `CREATE TABLE ... PRIMARY KEY / REFERENCES / NOT NULL`; documentos com `struct_pack`, `list` e `to_json`. As dicas dos exercícios explicam o que mais for preciso.

## 1 · Engenharia reversa: as cardinalidades reais
Para cada relacionamento abaixo, meça **quantas linhas do lado N existem para cada linha do lado 1** (mínimo e máximo). Com isso, classifique o relacionamento como 1:1, 1:N ou N:N.

| # | Relacionamento | Pergunta |
|---|---|---|
| a | `customers` → `orders` (por `customer_id`) | Quantos pedidos tem cada `customer_id`? |
| b | pessoa (`customer_unique_id`) → `orders` | Quantos pedidos tem cada pessoa? |
| c | `orders` → `items` | Quantos itens tem cada pedido? (inclua pedidos sem itens) |
| d | `orders` → `payments` | Quantos pagamentos tem cada pedido? |
| e | `orders` ↔ `reviews` | Quantas avaliações tem cada pedido? E em quantos pedidos aparece cada `review_id`? |
| f | `products` → `items` e `sellers` → `items` | Algum item aponta para produto ou vendedor inexistente? |

In [9]:
# Dica: use o padrão do aquecimento A.1 para cada relacionamento.
# Para incluir pedidos SEM itens, parta de orders e use LEFT JOIN (junção que mantém
# todas as linhas da tabela da esquerda, mesmo sem correspondência) e COUNT(i.order_id):
# SELECT o.order_id, COUNT(i.order_id) AS n FROM orders o LEFT JOIN items i USING (order_id) GROUP BY 1
# Para (f), use NOT EXISTS, como na Aula 1.
sql("""
WITH
ra  AS (SELECT c.customer_id, COUNT(o.order_id) AS n
        FROM customers c LEFT JOIN orders o USING (customer_id) GROUP BY 1),
rb  AS (SELECT c.customer_unique_id, COUNT(o.order_id) AS n
        FROM customers c LEFT JOIN orders o USING (customer_id) GROUP BY 1),
rc  AS (SELECT o.order_id, COUNT(i.order_id) AS n
        FROM orders o LEFT JOIN items i USING (order_id) GROUP BY 1),
rd  AS (SELECT o.order_id, COUNT(p.order_id) AS n
        FROM orders o LEFT JOIN payments p USING (order_id) GROUP BY 1),
re1 AS (SELECT o.order_id, COUNT(r.order_id) AS n
        FROM orders o LEFT JOIN reviews r USING (order_id) GROUP BY 1),
re2 AS (SELECT review_id, COUNT(DISTINCT order_id) AS n
        FROM reviews GROUP BY 1)
SELECT 'a) pedidos por customer_id' AS relacionamento, MIN(n) AS minimo, MAX(n) AS maximo, ROUND(AVG(n), 3) AS media,
       COUNT(*) FILTER (WHERE n = 0) AS com_zero, COUNT(*) FILTER (WHERE n > 1) AS com_mais_de_um FROM ra
UNION ALL SELECT 'b) pedidos por pessoa (customer_unique_id)', MIN(n), MAX(n), ROUND(AVG(n), 3),
       COUNT(*) FILTER (WHERE n = 0), COUNT(*) FILTER (WHERE n > 1) FROM rb
UNION ALL SELECT 'c) itens por pedido', MIN(n), MAX(n), ROUND(AVG(n), 3),
       COUNT(*) FILTER (WHERE n = 0), COUNT(*) FILTER (WHERE n > 1) FROM rc
UNION ALL SELECT 'd) pagamentos por pedido', MIN(n), MAX(n), ROUND(AVG(n), 3),
       COUNT(*) FILTER (WHERE n = 0), COUNT(*) FILTER (WHERE n > 1) FROM rd
UNION ALL SELECT 'e1) avaliações por pedido', MIN(n), MAX(n), ROUND(AVG(n), 3),
       COUNT(*) FILTER (WHERE n = 0), COUNT(*) FILTER (WHERE n > 1) FROM re1
UNION ALL SELECT 'e2) pedidos por review_id', MIN(n), MAX(n), ROUND(AVG(n), 3),
       COUNT(*) FILTER (WHERE n = 0), COUNT(*) FILTER (WHERE n > 1) FROM re2
""")

,relacionamento,minimo,maximo,media,com_zero,com_mais_de_um
0,a) pedidos por customer_id,1,1,1.000,0,0
1,b) pedidos por pessoa (customer_unique_id),1,17,1.035,0,2997
2,c) itens por pedido,0,21,1.133,775,9803
3,d) pagamentos por pedido,0,29,1.045,1,2961
4,e1) avaliações por pedido,0,3,0.998,768,547
5,e2) pedidos por review_id,1,3,1.008,0,789


In [10]:
# (f) integridade referencial de items
sql("""
SELECT
  (SELECT COUNT(*) FROM items i WHERE NOT EXISTS (SELECT 1 FROM products p WHERE p.product_id = i.product_id)) AS itens_sem_produto,
  (SELECT COUNT(*) FROM items i WHERE NOT EXISTS (SELECT 1 FROM sellers  s WHERE s.seller_id  = i.seller_id))  AS itens_sem_vendedor,
  (SELECT COUNT(*) FROM items i WHERE NOT EXISTS (SELECT 1 FROM orders   o WHERE o.order_id   = i.order_id))   AS itens_sem_pedido,
  (SELECT COUNT(*) FROM products p WHERE NOT EXISTS (SELECT 1 FROM items i WHERE i.product_id = p.product_id)) AS produtos_nunca_vendidos,
  (SELECT COUNT(*) FROM sellers  s WHERE NOT EXISTS (SELECT 1 FROM items i WHERE i.seller_id  = s.seller_id))  AS vendedores_sem_venda
""")

,itens_sem_produto,itens_sem_vendedor,itens_sem_pedido,produtos_nunca_vendidos,vendedores_sem_venda
0,0,0,0,0,0


### 1b · O grão de `items`
Uma linha de `items` representa uma unidade comprada ou uma linha de pedido com quantidade? Conte quantos pares (pedido, produto) aparecem em mais de uma linha e examine um pedido assim. Repare que não existe coluna de quantidade.

In [11]:
# Dica: GROUP BY order_id, product_id HAVING COUNT(*) > 1
# Para ver um exemplo, pegue um order_id do resultado e filtre items por ele.
display(sql("""
SELECT COUNT(*) AS pares_pedido_produto_repetidos, SUM(n) AS linhas_envolvidas, MAX(n) AS max_repeticoes
FROM (SELECT order_id, product_id, COUNT(*) AS n FROM items GROUP BY 1, 2 HAVING COUNT(*) > 1)
"""))

exemplo = contar("""
SELECT order_id FROM items GROUP BY order_id, product_id HAVING COUNT(*) > 1
ORDER BY COUNT(*) DESC LIMIT 1
""")
sql(f"""
SELECT order_id, order_item_id, product_id, seller_id, price, freight_value, shipping_limit_date
FROM items WHERE order_id = '{exemplo}' ORDER BY order_item_id
""")

,pares_pedido_produto_repetidos,linhas_envolvidas,max_repeticoes
0,7088,17313.0,20


,order_id,order_item_id,product_id,seller_id,price,freight_value,shipping_limit_date
0,ab14fdcfbe524636d65ee38360e22ce8,1,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
1,ab14fdcfbe524636d65ee38360e22ce8,2,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
2,ab14fdcfbe524636d65ee38360e22ce8,3,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
3,ab14fdcfbe524636d65ee38360e22ce8,4,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
4,ab14fdcfbe524636d65ee38360e22ce8,5,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
5,ab14fdcfbe524636d65ee38360e22ce8,6,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
6,ab14fdcfbe524636d65ee38360e22ce8,7,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
7,ab14fdcfbe524636d65ee38360e22ce8,8,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
8,ab14fdcfbe524636d65ee38360e22ce8,9,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23
9,ab14fdcfbe524636d65ee38360e22ce8,10,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,98.7,14.44,2017-08-30 14:30:23


In [12]:
sql(f"""
SELECT order_id, product_id, seller_id,
       COUNT(*)           AS quantidade,
       MIN(price)         AS preco_unitario,
       SUM(price)         AS valor_total,
       SUM(freight_value) AS frete_total
FROM items
WHERE order_id = '{exemplo}'
GROUP BY ALL
""")

,order_id,product_id,seller_id,quantidade,preco_unitario,valor_total,frete_total
0,ab14fdcfbe524636d65ee38360e22ce8,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,20,98.7,1974.0,288.8


In [13]:
sql(f"""
SELECT
  (SELECT SUM(price) + SUM(freight_value) FROM items    WHERE order_id = '{exemplo}') AS itens_mais_frete_somado,
  (SELECT SUM(price) + MAX(freight_value) FROM items    WHERE order_id = '{exemplo}') AS itens_mais_um_frete,
  (SELECT SUM(payment_value)              FROM payments WHERE order_id = '{exemplo}') AS total_pago
""")

,itens_mais_frete_somado,itens_mais_um_frete,total_pago
0,2262.8,1988.44,2262.8


In [14]:
sql("""
WITH i AS (SELECT order_id, SUM(price) + SUM(freight_value) AS esperado FROM items GROUP BY 1),
     p AS (SELECT order_id, SUM(payment_value) AS pago FROM payments GROUP BY 1)
SELECT COUNT(*) AS pedidos,
       COUNT(*) FILTER (WHERE abs(esperado - pago) < 0.01) AS bate_exato,
       ROUND(100.0 * COUNT(*) FILTER (WHERE abs(esperado - pago) < 0.01) / COUNT(*), 1) AS pct_bate
FROM i JOIN p USING (order_id)
""")

,pedidos,bate_exato,pct_bate
0,98665,98215,99.5


## 2 · Dependências funcionais e formas normais

### 2a · O endereço do vendedor (3FN)
Em `sellers`, a cidade e o estado estão guardados em cada vendedor. Se o **prefixo de CEP** determinasse o estado (`seller_zip_code_prefix → seller_state`), guardar o estado em cada vendedor seria uma **dependência transitiva** (`seller_id → CEP → estado`), o que viola a 3FN.
1. Teste a dependência `seller_zip_code_prefix → seller_state`. Quantos prefixos aparecem com mais de um estado?
2. Liste esses casos com as cidades e os estados. O que parece ter acontecido? Formule hipóteses, sem concluir.
3. Repita o teste 1 para `customers` (`customer_zip_code_prefix → customer_state` e `→ customer_city`).

In [15]:
# Dica: padrão do aquecimento A.2, trocando a coluna agrupada.
# Para listar: list(seller_state) e list(seller_city) agrupados por prefixo, filtrando os prefixos com violação.
# Item 1: o prefixo de CEP determina o estado?
display(sql("""
SELECT COUNT(*) AS prefixos_com_mais_de_um_estado
FROM (SELECT seller_zip_code_prefix FROM sellers GROUP BY 1 HAVING COUNT(DISTINCT seller_state) > 1)
"""))

# Item 2: os casos
sql("""
SELECT seller_zip_code_prefix,
       list(seller_state ORDER BY seller_state) AS estados,
       list(seller_city  ORDER BY seller_state) AS cidades,
       COUNT(*) AS vendedores
FROM sellers
GROUP BY 1
HAVING COUNT(DISTINCT seller_state) > 1
ORDER BY 1
""")

,prefixos_com_mais_de_um_estado
0,17


,seller_zip_code_prefix,estados,cidades,vendedores
0,21210,"[RJ, RN]","[rio de janeiro, rio de janeiro]",2
1,22783,"[RJ, SP]","[rio de janeiro, rio de janeiro]",2
2,31160,"[MG, SP]","[belo horizonte, belo horizonte]",2
3,36010,"[MG, MG, SP]","[juiz de fora, juiz de fora, juiz de fora]",3
4,37540,"[MG, MG, MG, MG, MG, MG, SP]","[santa rita do sapucai, santa rita do sapucai,...",7
5,37795,"[MG, SP]","[andradas, andradas]",2
6,44600,"[BA, SP]","[ipira, ipira]",2
7,80240,"[PR, PR, PR, SP]","[curitiba, curitiba, curitiba, curitiba]",4
8,81020,"[PR, SP]","[curitiba, curitiba]",2
9,81560,"[PR, SP]","[curitiba, curitiba]",2


In [16]:
# Item 3: o mesmo teste em customers
display(sql("""
SELECT
  (SELECT COUNT(DISTINCT customer_zip_code_prefix) FROM customers) AS total_prefixos,
  (SELECT COUNT(*) FROM (SELECT customer_zip_code_prefix FROM customers GROUP BY 1
                         HAVING COUNT(DISTINCT customer_state) > 1)) AS prefixos_com_mais_de_um_estado,
  (SELECT COUNT(*) FROM (SELECT customer_zip_code_prefix FROM customers GROUP BY 1
                         HAVING COUNT(DISTINCT customer_city) > 1))  AS prefixos_com_mais_de_uma_cidade
"""))

# Casos de prefixo com mais de um estado
display(sql("""
SELECT customer_zip_code_prefix,
       list(DISTINCT customer_state) AS estados,
       list(DISTINCT customer_city)  AS cidades,
       COUNT(*) AS clientes
FROM customers
GROUP BY 1
HAVING COUNT(DISTINCT customer_state) > 1
ORDER BY 1
"""))

# Casos de prefixo com mais de uma cidade (os 15 com mais cidades)
sql("""
SELECT customer_zip_code_prefix,
       list(DISTINCT customer_city)  AS cidades,
       list(DISTINCT customer_state) AS estados,
       COUNT(*) AS clientes
FROM customers
GROUP BY 1
HAVING COUNT(DISTINCT customer_city) > 1
ORDER BY COUNT(DISTINCT customer_city) DESC, clientes DESC
LIMIT 15
""")

,total_prefixos,prefixos_com_mais_de_um_estado,prefixos_com_mais_de_uma_cidade
0,14995,0,39


,customer_zip_code_prefix,estados,cidades,clientes


,customer_zip_code_prefix,cidades,estados,clientes
0,45816,"[arraial d'ajuda, arraial d ajuda, porto seguro]",[BA],10
1,13820,"[jaguariuna, monte alegre do sul]",[SP],60
2,23970,"[paraty, parati]",[RJ],32
3,37925,"[piumhii, piumhi]",[MG],31
4,13450,"[santa barbara d'oeste, santa barbara d oeste]",[SP],30
5,42850,"[dias d'avila, dias d avila]",[BA],25
6,13806,"[mogi mirim, mogi-mirim]",[SP],24
7,6823,"[embu das artes, embu]",[SP],20
8,13454,"[santa barbara d'oeste, santa barbara d oeste]",[SP],20
9,71065,"[brasilia, guara]",[DF],19


### 2b · O preço do item (2FN)
Em `items`, a chave é composta: (`order_id`, `order_item_id`). Se o preço dependesse **só do produto** (`product_id → price`), ele deveria morar em `products`, e não em `items`.
Teste a dependência `product_id → price`. Quantos produtos foram vendidos por mais de um preço? A modelagem atual está certa?

In [17]:
# Dica: GROUP BY product_id HAVING COUNT(DISTINCT price) > 1
# Teste da dependência product_id -> price
display(sql("""
WITH p AS (SELECT product_id, COUNT(*) AS vendas, COUNT(DISTINCT price) AS precos FROM items GROUP BY 1)
SELECT COUNT(*)                            AS produtos_vendidos,
       COUNT(*) FILTER (WHERE vendas > 1)  AS vendidos_mais_de_uma_vez,
       COUNT(*) FILTER (WHERE precos > 1)  AS com_mais_de_um_preco,
       ROUND(100.0 * COUNT(*) FILTER (WHERE precos > 1)
             / COUNT(*) FILTER (WHERE vendas > 1), 1) AS pct_dos_revendidos,
       MAX(precos)                         AS max_precos_distintos
FROM p
"""))

# Exemplos: produtos com maior variação de preço
display(sql("""
SELECT product_id, COUNT(*) AS vendas, COUNT(DISTINCT price) AS precos_distintos,
       MIN(price) AS menor, MAX(price) AS maior,
       ROUND(100.0 * (MAX(price) - MIN(price)) / MIN(price), 1) AS variacao_pct
FROM items GROUP BY 1
HAVING COUNT(DISTINCT price) > 1
ORDER BY precos_distintos DESC
LIMIT 10
"""))

# O preço depende do produto E do vendedor? (product_id, seller_id) -> price
sql("""
WITH p AS (SELECT product_id, seller_id, COUNT(DISTINCT price) AS precos FROM items GROUP BY 1, 2)
SELECT COUNT(*) AS pares_produto_vendedor,
       COUNT(*) FILTER (WHERE precos > 1) AS com_mais_de_um_preco
FROM p
""")

,produtos_vendidos,vendidos_mais_de_uma_vez,com_mais_de_um_preco,pct_dos_revendidos,max_precos_distintos
0,32951,14834,5900,39.8,31


,product_id,vendas,precos_distintos,menor,maior,variacao_pct
0,437c05a395e9e47f9762e677a7068ce7,155,31,44.98,59.90,33.2
1,d285360f29ac7fd97640bf0baef03de0,123,29,154.91,349.99,125.9
2,e0d64dcfaa3b6db5c54ca298ae101d05,194,27,124.90,255.61,104.7
3,656e0eca68dcecf6a31b8ececfabe3e8,141,23,65.00,109.90,69.1
4,6f3b5b605d91b7439c5e3f5a8dffeea7,117,23,128.90,239.90,86.1
5,e53e557d5a159f5aa2c5e995dfdf244b,183,21,74.90,109.90,46.7
6,69455f41626a745aea9ee9164cb9eafd,41,21,110.00,337.69,207.0
7,53b36df67ebb7c41585e8d54d6772e08,323,21,99.90,179.90,80.1
8,461f43be3bdf8844e65b62d9ac2c7a5a,146,20,115.00,227.00,97.4
9,4298b7e67dc399c200662b569563a2b2,88,20,119.00,179.90,51.2


,pares_produto_vendedor,com_mais_de_um_preco
0,34448,5435


### 2c · Provocando uma anomalia de atualização (2FN)
Junte `orders` e `items` numa tabela única e **desnormalizada** (`pedidos_itens`), com uma linha por item e o status do pedido repetido em cada linha.
1. Crie a tabela com `CREATE TABLE pedidos_itens AS SELECT ...`.
2. Simule um sistema que atualiza o status olhando só **uma** linha: mude para `'canceled'` apenas a linha `order_item_id = 1` de todos os pedidos com mais de um item.
3. Conte quantos pedidos agora têm mais de um status. Qual dependência causou o problema e qual forma normal ela viola?

In [19]:
# Dica:
# CREATE OR REPLACE TABLE pedidos_itens AS
#   SELECT i.*, o.order_status FROM items i JOIN orders o USING (order_id)
# UPDATE pedidos_itens SET order_status = 'canceled'
#   WHERE order_item_id = 1 AND order_id IN (SELECT order_id FROM pedidos_itens GROUP BY 1 HAVING COUNT(*) > 1)
# Item 1: tabela desnormalizada, com o status do pedido repetido em cada item
con.execute("""
CREATE OR REPLACE TABLE pedidos_itens AS
SELECT i.*, o.order_status FROM items i JOIN orders o USING (order_id)
""")

q_inconsistentes = """SELECT COUNT(*) FROM (SELECT order_id FROM pedidos_itens
                      GROUP BY 1 HAVING COUNT(DISTINCT order_status) > 1)"""
q_afetados = """SELECT COUNT(*) FROM (SELECT order_id FROM pedidos_itens
                GROUP BY 1 HAVING COUNT(*) > 1)"""

print("Linhas na tabela:", contar("SELECT COUNT(*) FROM pedidos_itens"))
print("Pedidos com mais de um item (alvo do UPDATE):", contar(q_afetados))
print("Pedidos com mais de um status ANTES:", contar(q_inconsistentes))

# Item 2: o sistema atualiza só a linha do primeiro item
con.execute("""
UPDATE pedidos_itens SET order_status = 'canceled'
WHERE order_item_id = 1
  AND order_id IN (SELECT order_id FROM pedidos_itens GROUP BY 1 HAVING COUNT(*) > 1)
""")

# Item 3: o estrago
print("Pedidos com mais de um status DEPOIS:", contar(q_inconsistentes))

exemplo = contar("""SELECT order_id FROM pedidos_itens
                    GROUP BY 1 HAVING COUNT(DISTINCT order_status) > 1 LIMIT 1""")
sql(f"""
SELECT order_id, order_item_id, product_id, price, order_status
FROM pedidos_itens WHERE order_id = '{exemplo}' ORDER BY order_item_id
""")

Linhas na tabela: 112650
Pedidos com mais de um item (alvo do UPDATE): 9803
Pedidos com mais de um status ANTES: 0
Pedidos com mais de um status DEPOIS: 9762


,order_id,order_item_id,product_id,price,order_status
0,88d1a4053f36cee7a806c473458c6c11,1,af2d506fb01b724e03591349ad36b67e,48.9,canceled
1,88d1a4053f36cee7a806c473458c6c11,2,ec9ec25e8e942a90369022c80ec9abe4,48.9,delivered


## 3 · Restrições: o banco como guardião
Os arquivos CSV não impõem nenhuma regra. Vamos criar um esquema com PK e FK e carregar os dados nele.
1. Crie `t_customers` (PK `customer_id`) e `t_orders` (PK `order_id`, FK `customer_id` → `t_customers`). Carregue ambas. Corrija o CEP do cliente com `lpad` (preenche com zeros à esquerda), como visto na Aula 1.
2. Crie `t_reviews` com PK em `review_id` e FK `order_id` → `t_orders`. Tente carregar. O que acontece? Quantas linhas entram?
3. Proponha e implemente uma chave que funcione para `t_reviews`. O que ela revela sobre a cardinalidade entre pedidos e avaliações?
4. Prove que a FK protege: tente inserir em `t_orders` um pedido com `customer_id` inexistente.
5. Por que não é possível criar uma PK em `geolocation` pelo prefixo de CEP? Proponha, em texto, uma tabela que permita essa PK.

In [20]:
# Dica: modelo de DDL
# con.execute("""CREATE OR REPLACE TABLE t_customers (
#     customer_id VARCHAR PRIMARY KEY, customer_unique_id VARCHAR NOT NULL,
#     zip_prefix VARCHAR(5), city VARCHAR, state VARCHAR(2))""")
# con.execute("INSERT INTO t_customers SELECT customer_id, customer_unique_id, "
#             "lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0'), customer_city, customer_state FROM customers")
# Na FK: order_id VARCHAR REFERENCES t_orders(order_id)
# Use tentar(...) para os comandos que podem ser recusados.
# Atenção: no DuckDB, crie as tabelas na ordem pai -> filho e, se precisar recriar,
# apague primeiro os filhos (DROP TABLE IF EXISTS t_reviews).
# Para recriar, apague primeiro os filhos
for t in ["t_reviews", "t_orders", "t_customers"]:
    con.execute(f"DROP TABLE IF EXISTS {t}")

con.execute("""
CREATE TABLE t_customers (
    customer_id        VARCHAR PRIMARY KEY,
    customer_unique_id VARCHAR NOT NULL,
    zip_prefix         VARCHAR(5) NOT NULL,
    city               VARCHAR,
    state              VARCHAR(2) NOT NULL
)""")
tentar("""
INSERT INTO t_customers
SELECT customer_id, customer_unique_id,
       lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0'), customer_city, customer_state
FROM customers""")

con.execute("""
CREATE TABLE t_orders (
    order_id                VARCHAR PRIMARY KEY,
    customer_id             VARCHAR NOT NULL REFERENCES t_customers(customer_id),
    order_status            VARCHAR NOT NULL,
    purchase_ts             TIMESTAMP NOT NULL,
    approved_at             TIMESTAMP,
    delivered_carrier_date  TIMESTAMP,
    delivered_customer_date TIMESTAMP,
    estimated_delivery_date TIMESTAMP
)""")
tentar("""
INSERT INTO t_orders
SELECT order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at,
       order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date
FROM orders""")

sql("SELECT (SELECT COUNT(*) FROM t_customers) AS clientes, (SELECT COUNT(*) FROM t_orders) AS pedidos")

OK: comando executado.
OK: comando executado.


,clientes,pedidos
0,99441,99441


In [21]:
con.execute("""
CREATE TABLE t_reviews (
    review_id              VARCHAR PRIMARY KEY,
    order_id               VARCHAR NOT NULL REFERENCES t_orders(order_id),
    review_score           INTEGER,
    review_comment_title   VARCHAR,
    review_comment_message VARCHAR,
    review_creation_date   TIMESTAMP,
    review_answer_ts       TIMESTAMP
)""")
tentar("""
INSERT INTO t_reviews
SELECT review_id, order_id, review_score, review_comment_title, review_comment_message,
       review_creation_date, review_answer_timestamp
FROM reviews""")
sql("SELECT COUNT(*) AS linhas_carregadas FROM t_reviews")

RECUSADO PELO BANCO: ConstraintException
Constraint Error: PRIMARY KEY or UNIQUE constraint violation: duplicate key "3242cc306a9218d0377831e175d62fbf"


,linhas_carregadas
0,0


In [22]:
con.execute("DROP TABLE IF EXISTS t_reviews")
con.execute("""
CREATE TABLE t_reviews (
    review_id              VARCHAR NOT NULL,
    order_id               VARCHAR NOT NULL REFERENCES t_orders(order_id),
    review_score           INTEGER,
    review_comment_title   VARCHAR,
    review_comment_message VARCHAR,
    review_creation_date   TIMESTAMP,
    review_answer_ts       TIMESTAMP,
    PRIMARY KEY (review_id, order_id)
)""")
tentar("""
INSERT INTO t_reviews
SELECT review_id, order_id, review_score, review_comment_title, review_comment_message,
       review_creation_date, review_answer_timestamp
FROM reviews""")
sql("SELECT (SELECT COUNT(*) FROM reviews) AS bronze, (SELECT COUNT(*) FROM t_reviews) AS carregadas")

OK: comando executado.


,bronze,carregadas
0,99224,99224


In [23]:
tentar("""
INSERT INTO t_orders (order_id, customer_id, order_status, purchase_ts)
VALUES ('pedido_teste', 'cliente_inexistente', 'created', TIMESTAMP '2018-01-01 10:00:00')
""")
sql("SELECT COUNT(*) AS pedidos_teste_gravados FROM t_orders WHERE order_id = 'pedido_teste'")

RECUSADO PELO BANCO: ConstraintException
Constraint Error: Violates foreign key constraint because key "customer_id: cliente_inexistente" does not exist in the referenced table


,pedidos_teste_gravados
0,0


In [24]:
display(sql("""
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT geolocation_zip_code_prefix) AS prefixos,
       ROUND(COUNT(*) / COUNT(DISTINCT geolocation_zip_code_prefix), 1) AS linhas_por_prefixo,
       (SELECT COUNT(*) FROM (SELECT geolocation_zip_code_prefix FROM geolocation GROUP BY 1
                              HAVING COUNT(DISTINCT geolocation_state) > 1)) AS prefixos_com_mais_de_um_estado
FROM geolocation
"""))

# Proposta: uma linha por prefixo
con.execute("DROP TABLE IF EXISTS t_cep")
con.execute("""
CREATE TABLE t_cep (
    zip_prefix VARCHAR(5) PRIMARY KEY,
    lat        DOUBLE,
    lng        DOUBLE,
    city       VARCHAR,
    state      VARCHAR(2),
    pontos     INTEGER
)""")
tentar("""
INSERT INTO t_cep
SELECT lpad(CAST(geolocation_zip_code_prefix AS VARCHAR), 5, '0'),
       median(geolocation_lat), median(geolocation_lng),
       mode(geolocation_city), mode(geolocation_state), COUNT(*)
FROM geolocation
GROUP BY 1""")
sql("SELECT COUNT(*) AS prefixos_na_t_cep FROM t_cep")

,linhas,prefixos,linhas_por_prefixo,prefixos_com_mais_de_um_estado
0,1000163,19015,52.6,8


OK: comando executado.


,prefixos_na_t_cep
0,19015


## 4 · O pedido como documento (NoSQL)
Num banco de documentos, a pergunta de modelagem muda: em vez de "como evitar redundância?", passa a ser "como a aplicação lê o dado?". Se a tela do pedido mostra cliente, itens e pagamentos juntos, faz sentido **embutir** tudo num só documento.
1. Crie a tabela `pedidos_doc` com uma linha por pedido e uma coluna `doc` em JSON contendo: dados do pedido, um objeto `cliente` e as listas `itens` e `pagamentos`.
2. Mostre o documento de um pedido com 3 itens, formatado.
3. Consultando **só os documentos**, calcule o total de itens e o faturamento (soma dos preços). Confira contra a tabela `items`.
4. Quantos documentos têm a lista de itens vazia ou nula? De onde vêm?
5. Qual o tamanho total dos documentos em MB? Compare com o tamanho dos CSVs de `orders`, `customers`, `items` e `payments` juntos.

In [25]:
# Dica: para cada pedido, uma subconsulta monta a lista de itens:
# (SELECT list(struct_pack(item := i.order_item_id, produto := i.product_id, preco := i.price)
#          ORDER BY i.order_item_id)
#    FROM items i WHERE i.order_id = o.order_id) AS itens
# O documento inteiro: to_json(struct_pack(order_id := o.order_id, ..., cliente := struct_pack(...), itens := (...)))
# Para somar preços dentro do JSON: FROM pedidos_doc, json_each(doc, '$.itens') x  e  x.value ->> 'preco'
# (json_each "desdobra" a lista em linhas; ->> extrai um campo como texto; converta com CAST(... AS DOUBLE))
# Tamanho dos arquivos: os.path.getsize(caminho) em bytes.
con.execute("""
CREATE OR REPLACE TABLE pedidos_doc AS
WITH it AS (
  SELECT order_id,
         list(struct_pack(item := order_item_id, produto := product_id, vendedor := seller_id,
                          preco := price, frete := freight_value) ORDER BY order_item_id) AS itens
  FROM items GROUP BY 1
),
pg AS (
  SELECT order_id,
         list(struct_pack(sequencial := payment_sequential, tipo := payment_type,
                          parcelas := payment_installments, valor := payment_value)
              ORDER BY payment_sequential) AS pagamentos
  FROM payments GROUP BY 1
)
SELECT o.order_id,
       to_json(struct_pack(
         order_id      := o.order_id,
         status        := o.order_status,
         comprado_em   := o.order_purchase_timestamp,
         aprovado_em   := o.order_approved_at,
         entregue_em   := o.order_delivered_customer_date,
         previsto_para := o.order_estimated_delivery_date,
         cliente := struct_pack(
             customer_id := c.customer_id,
             pessoa      := c.customer_unique_id,
             cep         := lpad(CAST(c.customer_zip_code_prefix AS VARCHAR), 5, '0'),
             cidade      := c.customer_city,
             uf          := c.customer_state),
         itens      := it.itens,
         pagamentos := pg.pagamentos
       )) AS doc
FROM orders o
LEFT JOIN customers c USING (customer_id)
LEFT JOIN it USING (order_id)
LEFT JOIN pg USING (order_id)
""")
sql("SELECT COUNT(*) AS documentos FROM pedidos_doc")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,documentos
0,99441


In [26]:
doc = contar("SELECT doc FROM pedidos_doc WHERE json_array_length(doc, '$.itens') = 3 LIMIT 1")
print(json.dumps(json.loads(doc), indent=2, ensure_ascii=False))

{
  "order_id": "a0ed7e292495e32e3a7e3ae479219c4d",
  "status": "delivered",
  "comprado_em": "2017-09-12 18:23:45",
  "aprovado_em": "2017-09-12 18:35:15",
  "entregue_em": "2017-09-25 17:27:01",
  "previsto_para": "2017-10-02 00:00:00",
  "cliente": {
    "customer_id": "3d2110fea7787994f7ff93361842685e",
    "pessoa": "a573d37f0b329d5fab25d17e3c81482d",
    "cep": "20021",
    "cidade": "rio de janeiro",
    "uf": "RJ"
  },
  "itens": [
    {
      "item": 1,
      "produto": "a4c7e73a8ce075149b5f6592fdd29367",
      "vendedor": "87142160b41353c4e5fca2360caf6f92",
      "preco": 149.0,
      "frete": 18.32
    },
    {
      "item": 2,
      "produto": "a4c7e73a8ce075149b5f6592fdd29367",
      "vendedor": "87142160b41353c4e5fca2360caf6f92",
      "preco": 149.0,
      "frete": 18.32
    },
    {
      "item": 3,
      "produto": "a4c7e73a8ce075149b5f6592fdd29367",
      "vendedor": "87142160b41353c4e5fca2360caf6f92",
      "preco": 149.0,
      "frete": 18.32
    }
  ],
  "pagamento

In [30]:
sql("""
SELECT 'documentos' AS fonte,
       COUNT(x.value ->> 'preco') AS itens,
       ROUND(SUM(CAST(x.value ->> 'preco' AS DOUBLE)), 2) AS faturamento
FROM pedidos_doc, json_each(doc, '$.itens') AS x
UNION ALL
SELECT 'tabela items', COUNT(*), ROUND(SUM(price), 2) FROM items
""")

,fonte,itens,faturamento
0,documentos,112650,13591643.7
1,tabela items,112650,13591643.7


In [31]:
sql("""
SELECT json_extract_string(doc, '$.status') AS status,
       COUNT(*) AS documentos_sem_itens
FROM pedidos_doc
WHERE coalesce(json_array_length(doc, '$.itens'), 0) = 0
GROUP BY 1
ORDER BY 2 DESC
""")

,status,documentos_sem_itens
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [32]:
mb_docs = contar("SELECT SUM(strlen(CAST(doc AS VARCHAR))) FROM pedidos_doc") / 1024**2
arqs = ["olist_orders_dataset.csv", "olist_customers_dataset.csv",
        "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv"]
mb_csv = sum(os.path.getsize(f"{DATA}/{a}") for a in arqs) / 1024**2

print(f"Documentos JSON: {mb_docs:,.1f} MB")
print(f"4 CSVs juntos:   {mb_csv:,.1f} MB")
print(f"Razão:           {mb_docs / mb_csv:,.2f}x")

Documentos JSON: 57.2 MB
4 CSVs juntos:   44.7 MB
Razão:           1.28x


### 4b · Embutir tem preço
No documento, os dados do cliente ficam **dentro de cada pedido**.
1. Quantas pessoas (`customer_unique_id`) aparecem com mais de um prefixo de CEP?
2. Se uma pessoa muda de endereço, quantos documentos precisam ser alterados num modelo que embute o cliente? E no modelo relacional?
3. **Hipótese a discutir, não a concluir:** por que a Olist criaria um `customer_id` novo a cada pedido? Que vantagem isso daria ao registrar o endereço de entrega?

In [33]:
# Dica: GROUP BY customer_unique_id HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1
# Item 1: pessoas com mais de um prefixo de CEP
display(sql("""
SELECT COUNT(*) AS pessoas_com_mais_de_um_cep
FROM (SELECT customer_unique_id FROM customers GROUP BY 1
      HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1)
"""))

# Item 2: quantos documentos cada mudança de endereço afetaria
display(sql("""
WITH p AS (
  SELECT customer_unique_id,
         COUNT(*) AS pedidos,
         COUNT(DISTINCT customer_zip_code_prefix) AS ceps
  FROM customers
  GROUP BY 1
  HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1
)
SELECT COUNT(*)      AS pessoas,
       SUM(pedidos)  AS documentos_com_o_cliente_embutido,
       MAX(pedidos)  AS max_documentos_de_uma_pessoa,
       MAX(ceps)     AS max_ceps_de_uma_pessoa
FROM p
"""))

# Exemplo: o histórico de endereços de uma pessoa
pessoa = contar("""
SELECT customer_unique_id FROM customers GROUP BY 1
HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1
ORDER BY COUNT(DISTINCT customer_zip_code_prefix) DESC, COUNT(*) DESC LIMIT 1
""")
sql(f"""
SELECT c.customer_unique_id, c.customer_id, o.order_purchase_timestamp,
       lpad(CAST(c.customer_zip_code_prefix AS VARCHAR), 5, '0') AS cep,
       c.customer_city, c.customer_state
FROM customers c JOIN orders o USING (customer_id)
WHERE c.customer_unique_id = '{pessoa}'
ORDER BY o.order_purchase_timestamp
""")

,pessoas_com_mais_de_um_cep
0,250


,pessoas,documentos_com_o_cliente_embutido,max_documentos_de_uma_pessoa,max_ceps_de_uma_pessoa
0,250,535.0,9,3


,customer_unique_id,customer_id,order_purchase_timestamp,cep,customer_city,customer_state
0,3e43e6105506432c953e165fb2acf44c,06ec7aed3161859018db02f695b6600d,2017-09-18 18:53:15,11701,praia grande,SP
1,3e43e6105506432c953e165fb2acf44c,a4413edae691fd0d28bd04b8a38b259d,2017-12-01 09:30:36,11700,praia grande,SP
2,3e43e6105506432c953e165fb2acf44c,b70f8552b91ef49129519206966e2472,2017-12-01 22:37:41,11700,praia grande,SP
3,3e43e6105506432c953e165fb2acf44c,e68e6423401e85c138229b23d4bf4761,2017-12-01 22:37:42,11700,praia grande,SP
4,3e43e6105506432c953e165fb2acf44c,b63c9cfc5dcb06ed307ca95957fa216c,2018-01-11 10:56:15,11704,praia grande,SP
5,3e43e6105506432c953e165fb2acf44c,2bbd32d4ef14893d2d8c1a0df08403cf,2018-02-12 10:12:54,11701,praia grande,SP
6,3e43e6105506432c953e165fb2acf44c,6b92dda6e324283dbe4969c443e3bf1a,2018-02-12 10:12:55,11701,praia grande,SP
7,3e43e6105506432c953e165fb2acf44c,f37a14140ac3a8efbb4d2eab6585edb7,2018-02-20 10:45:22,11701,praia grande,SP
8,3e43e6105506432c953e165fb2acf44c,1815fdaaae692479110428598596b5e5,2018-02-27 18:36:39,11701,praia grande,SP


## 5 · Qual NoSQL para qual necessidade? (discussão, sem código)
Para cada necessidade, indique o tipo de banco mais adequado (**documento**, **chave-valor**, **colunar** ou **grafo**) e justifique em uma frase.

| Necessidade | Tipo | Justificativa |
|---|---|---|
| Consultar o status atual de um pedido pelo `order_id`, milhares de vezes por segundo | **Chave-valor** | O acesso é sempre por uma chave única e não precisa de nenhum JOIN; bancos chave-valor (Redis, DynamoDB) entregam esse tipo de leitura em milissegundos e com altíssima vazão. |
| Catálogo de produtos em que cada categoria tem atributos diferentes | **Documento** | Cada documento tem sua própria estrutura, então uma geladeira (voltagem, litros) e um livro (autor, páginas) convivem sem dezenas de colunas nulas (MongoDB). |
| Recomendação "quem comprou este produto também comprou..." | **Grafo** | A pergunta é uma travessia produto → clientes que compraram → outros produtos, natural num grafo (Neo4j) e cara em SQL, que exigiria vários JOINs encadeados. |
| Gravar bilhões de eventos de rastreamento de entrega, lidos por pedido e por data | **Colunar (wide-column)** | Cassandra e HBase são feitos para escrita massiva; com `order_id` como chave de partição e a data como chave de ordenação, ler os eventos de um pedido num período é uma consulta direta. |

> **Observação:** aqui "colunar" é o modelo NoSQL *wide-column* (Cassandra, HBase). É diferente do armazenamento colunar analítico (Parquet, DuckDB), que serve para agregações em grande volume, e não para gravar eventos um a um.

| Tabela | PK | FKs | Grão (uma linha representa...) | Ajuste em relação ao CSV original |
|---|---|---|---|---|
| customers | `customer_id` | `zip_prefix` → `t_cep` (após tratar 279 CEPs sem correspondência) | O cliente **num pedido**, com o endereço de entrega daquela compra (1:1 com orders) | CEP como texto de 5 dígitos; a pessoa é `customer_unique_id` (250 pessoas usaram mais de um CEP) e pode virar tabela própria |
| orders | `order_id` | `customer_id` → customers | Um pedido | Datas como TIMESTAMP; status `NOT NULL` e mantido só aqui (o 2c mostrou a anomalia ao repeti-lo nos itens) |
| items | (`order_id`, `order_item_id`) | `order_id` → orders; `product_id` → products; `seller_id` → sellers | **Uma unidade** vendida; quantidade = `COUNT(*)` (7.088 pares pedido-produto repetidos) | Preço e frete ficam no item: são por unidade (99,5% dos pedidos fecham com o pagamento) e variam por venda (39,8% dos produtos revendidos têm mais de um preço) |
| payments | (`order_id`, `payment_sequential`) | `order_id` → orders | Um meio de pagamento usado no pedido (até 29 por pedido) | Nenhum estrutural |
| reviews | (`review_id`, `order_id`) | `order_id` → orders | A associação de uma avaliação a um pedido | PK em `review_id` foi recusada (N:N); ideal separar `avaliacoes` (PK `review_id`) e tabela associativa `avaliacao_pedido` |
| products | `product_id` | `product_category_name` → category_translation | Um produto do catálogo | Peso 0 como nulo; corrigir colunas com erro de digitação (`lenght`); catálogo contém só produtos vendidos (viés de seleção) |
| sellers | `seller_id` | `zip_prefix` → `t_cep` | Um vendedor | CEP como texto; estado derivado do CEP, porque 17 prefixos têm estado divergente (dependência transitiva, viola a 3FN) |
| geolocation | sem PK no CSV (52,6 linhas por prefixo); proposta `t_cep` com PK `zip_prefix` | — | No CSV: um ponto observado. Na `t_cep`: um prefixo de CEP | Agregar por prefixo (mediana das coordenadas; cidade e UF mais frequentes, o que resolve os 8 prefixos com mais de um estado) |